In [0]:


%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%python
df = (spark
      .read
      .format("csv")
      .option("header",True)
      .option("inferSchema",True)
      .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))
df.display()

In [0]:
%python
df.createOrReplaceTempView("csv_view")
# df.createOrReplaceGlobalTempView("csv_view_global")


In [0]:
%python
display(spark.sql("select * from csv_view"))

In [0]:
from pyspark.sql import Row

data = [
    (1, "John", "US", 1200.50, "2026-01-15"),
    (2, "Alice", "UK", 850.75, "2026-02-10"),
    (3, "David", "FR", 1500.00, "2026-03-05"),
]

columns = [
    "customer_id",
    "customer_name",
    "country",
    "sales_amount",
    "order_date"
]

df1 = spark.createDataFrame(data, columns)

df1.createOrReplaceTempView("df1")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS learnspark.raw.orders
          (
              customer_id INT,
              customer_name STRING,
              country STRING,
              sales_amount DOUBLE,
              order_date STRING
          )
          """)

In [0]:
spark.sql("""
          INSERT INTO learnspark.raw.orders
          SELECT * FROM df1
          """)

In [0]:
spark.sql("select * from learnspark.raw.orders").display()

In [0]:
spark.sql("""DELETE FROM learnspark.raw.orders where customer_id in (2) and sales_amount = '960.75'""")

In [0]:
from pyspark.sql import Row

data = [
    (4, "Ajay", "DE", 1590.00, "2026-12-15"),
    (2, "Alice", "UK", 960.75, "2026-02-10"),
    (5, "David", "IT", 1250.00, "2026-10-05"),
]

columns = [
    "customer_id",
    "customer_name",
    "country",
    "sales_amount",
    "order_date"
]

df2 = spark.createDataFrame(data, columns)

df1.createOrReplaceTempView("df2")

### MergeInto Statement
- To use mergeInto statement, one table should persist in unity catalog

In [0]:
(df2.alias("src").mergeInto("learnspark.raw.orders",col("src.customer_id") == col("learnspark.raw.orders.customer_id"))
 .whenMatched().updateAll()
 .whenNotMatched().insertAll()
 .merge()
)
 

In [0]:
%sql
EXPLAIN select * from learnspark.raw.orders

```MERGE INTO target_table AS target USING source_table AS source ON target.id = source.id WHEN MATCHED THEN UPDATE SET target.name = source.name, target.salary = source.salary WHEN NOT MATCHED THEN INSERT (id, name, salary)VALUES (source.id, source.name, source.salary);```

### SQL Scripting Statements

#### For 

In [0]:
%sql
BEGIN
    DECLARE sum INT DEFAULT 0;

    sumNumbers: FOR row AS
        SELECT num FROM range(1, 20) AS t(num)
    DO
        IF row.num > 10 THEN
            LEAVE sumNumbers;

        ELSEIF row.num % 2 = 0 THEN
            ITERATE sumNumbers;
        END IF;

        SET sum = sum + row.num;
    END FOR sumNumbers;

    SELECT sum;
END;

In [0]:
%sql
BEGIN
    DECLARE choice DOUBLE DEFAULT 3.9;
    DECLARE result STRING;
    IF choice < 2 THEN
        VALUES ('one fish');
    ELSEIF choice < 3 THEN
        VALUES ('two fish');
    ELSEIF choice < 4 THEN
        VALUES ('red fish');
    ELSEIF choice < 5 OR choice IS NULL THEN
        VALUES ('blue fish');
    ELSE 
         VALUES ('no fish');
    END IF;
END;

    

#### While

In [0]:
%sql
BEGIN
    DECLARE sum INT DEFAULT 0;
    DECLARE num INT DEFAULT 0;
    sumNumbers: WHILE num < 10 
    DO
    SET num = num +1;
    IF num % 2 = 0 THEN
        ITERATE sumNumbers;
    END IF;
    SET sum = sum + num;
    END WHILE sumNumbers;
    SELECT sum;
END;


### SPARKSQL Auxiliary Statements

In [0]:
%sql
DESCRIBE DATABASE learnspark.raw

In [0]:
%sql
DESCRIBE TABLE learnspark.raw.orders

In [0]:
%sql
DESCRIBE QUERY SELECT * FROM learnspark.raw.orders

#### REFRESH

```REFRESH TABLE learnspark.raw.orders```

In [0]:
%sql
SHOW DATABASES;

SHOW SCHEMAS;

In [0]:
%sql
SHOW TABLES FROM learnspark.raw

In [0]:
%sql
USE learnspark.raw;
SHOW TABLE EXTENDED LIKE 'orders'

In [0]:
%sql
SHOW TBLPROPERTIES learnspark.raw.orders

In [0]:
%sql
SHOW PARTITIONS learnspark.raw.orders

#### SPARKSQL Advanced Functions

In [0]:
#### Aggregate Functions

In [0]:
df.createOrReplaceTempView("sql_tbl")

In [0]:
display(df.groupBy("customer_id").agg(array_agg("order_id")))

In [0]:
%sql
select customer_id, array_agg(order_id) from sql_tbl group by customer_id

In [0]:
%sql
SELECT array_agg(col) as array_col FROM (VALUES (1),(2),(3),(4)) AS tab(col)

In [0]:
%sql
SELECT customer_id,collect_list(product_id),collect_set(product_id) FROM sql_tbl group by customer_id

In [0]:
%sql
SELECT 
corr(price,quantity),
max(price),
min(price),
avg(price),
median(price)
FROM sql_tbl


#### Struct and Map

In [0]:
%sql
SELECT struct(1,2,3,"abc")

In [0]:
%sql
SELECT named_struct("a",1,"b",2)

In [0]:
%sql
SELECT map("a",1,"b",2,"c",3)

In [0]:
%sql
SELECT map_values(map("a",1,"b",2,"c",3));


In [0]:
%sql
SELECT map_keys(map("a",1,"b",2,"c",3));

### Date Functions

In [0]:
%sql
SELECT current_timestamp();
SELECT add_months(current_timestamp(),1);
SELECT add_months(current_timestamp(),-1);

In [0]:
%sql
SELECT unix_timestamp()

In [0]:
%sql
select date_from_unix_date(0)

In [0]:
%sql
SELECT dayname(current_timestamp()),
dayofmonth(current_timestamp()),
dayofmonth(current_timestamp())

In [0]:
%sql
SELECT extract(year from current_timestamp()),
extract(month from current_timestamp()),
extract(day from current_timestamp()),
extract(minute from current_timestamp())

### Window Functions

In [0]:
%sql
SELECT 
order_id,
LAG(order_date,1,'1900-01-01') over(order by order_id) as prev_order_date,
order_date,
LEAD(order_date,1,'9999-01-01') over(order by order_id) as next_order_date
FROM
sql_tbl

In [0]:
%sql
SELECT 
order_id,
customer_id,
order_status,
ntile(3) over( partition by order_status order by order_id) from sql_tbl

### Array Function

In [0]:
%sql
SELECT array(1,2,3);

In [0]:
%sql
SELECT array_append(array(1,2,3),9)

In [0]:
%sql
SELECT array_distinct(array(1,2,3,3))

In [0]:
%sql
SELECT array_compact(array(1,null,3,null,null))

In [0]:
%sql
SELECT array_insert(array(1,2,3),1,99);
SELECT array_insert(array(1,2,3),2,99);

In [0]:
%sql
SELECT array_prepend(array(1,2,3),0);
SELECT array_position(array(1,2,3),0)

### UDFs In SparkSQL

In [0]:
def my_upper_function(input:str )-> str:
    return input.upper()


In [0]:
spark.udf.register("my_upper",my_upper_function)

In [0]:
%sql
SELECT my_upper("hello")

### SPARKSQL Query Files

#### Reading CSV

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_csv
USING CSV
OPTIONS(
    path "/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv",
    header "true",
    inferSchema "true"
);
SELECT * FROM orders_csv


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_json
USING org.apache.spark.sql.json
OPTIONS(
    path "/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json",
    header "true",
    inferSchema "true"
);
SELECT * FROM orders_json


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_json
USING JSON
OPTIONS(
    path "/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json",
    header "true",
    inferSchema "true"
);
SELECT * FROM orders_json


### SparkSQL using DF

In [0]:
display(spark.sql("SELECT * FROM {df} where price > 10",df =df))

### Using File Format Connector

In [0]:
spark.sql("SELECT * from json.`/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json`").display()